## 4. Register CSV Data With Athena

Code learned and followed from AAI 540 Lab 2

In [2]:
#!pip install -q PyAthena

In [2]:
import boto3
import sagemaker
import pandas as pd
sess = sagemaker.Session()
bucket = sess.default_bucket()
role = sagemaker.get_execution_role()
region = boto3.Session().region_name

In [3]:
df = pd.read_csv(f's3://{bucket}/CCPP/data/feature_set/clean_data.csv')

In [4]:
df

,CDP,GTEP,TIT,AFDP,solarradiation,solarenergy,uvindex,AP,cloudcover,sealevelpressure,Target
0,10.564,19.166,1043.6,2.3927,0.0,0.0,0.0,1020.1,90.0,1023.0,110.16
1,10.572,19.119,1039.9,2.3732,0.0,0.0,0.0,1019.9,89.0,1023.0,109.23
2,10.543,19.178,1041.0,2.3854,0.0,0.0,0.0,1019.8,91.2,1023.1,109.62
3,10.458,19.180,1037.1,2.3825,0.0,0.0,0.0,1019.3,90.3,1022.3,108.88
4,10.464,19.206,1043.5,2.4044,0.0,0.0,0.0,1019.1,90.3,1022.2,110.09
...,...,...,...,...,...,...,...,...,...,...,...
14759,10.411,19.087,1037.0,3.1661,0.0,0.0,0.0,1028.5,78.2,1029.6,109.08
14760,10.344,19.016,1037.6,3.1923,0.0,0.0,0.0,1028.6,78.2,1028.8,108.79
14761,10.462,18.857,1038.0,3.3128,0.0,0.0,0.0,1028.5,46.0,1028.7,107.81
14762,11.771,23.563,1076.9,3.9831,0.0,0.0,0.0,1028.7,46.0,1028.7,131.41


In [5]:
from pyathena import connect

In [6]:
s3_private_path_ccpp_sv =f's3://{bucket}/CCPP/data/feature_set'

In [7]:
# Set S3 staging directory -- this is a temporary directory used for Athena queries
s3_staging_dir = "s3://{0}/athena/staging".format(bucket)

In [8]:
# Set Athena parameters
database_name = "ccpp_aws_fp"
table_name_csv = "final_data_csv"

In [9]:
conn = connect(region_name=region, s3_staging_dir=s3_staging_dir)

In [10]:
# SQL statement to execute
statement = """CREATE EXTERNAL TABLE IF NOT EXISTS {}.{}(
         gt_comp_dis_pressure float,
         gt_exhaust_pressure float,
         gt_inlet_temp float,
         gt_air_filter_diff_pressure float,
         solar_radiation float,
         solar_energy float,
         uv_index float,
         gt_ambient_pressure float,
         cloud_cover float,
         sea_level_pressure  float,
         gt_energy_yield float
) ROW FORMAT DELIMITED FIELDS TERMINATED BY ',' LINES TERMINATED BY '\\n' LOCATION '{}'
TBLPROPERTIES ('skip.header.line.count'='1')""".format(
    database_name, table_name_csv, s3_private_path_ccpp_sv
)
# Print SQL Query
print(statement)

CREATE EXTERNAL TABLE IF NOT EXISTS ccpp_aws_fp.final_data_csv(
         gt_comp_dis_pressure float,
         gt_exhaust_pressure float,
         gt_inlet_temp float,
         gt_air_filter_diff_pressure float,
         solar_radiation float,
         solar_energy float,
         uv_index float,
         gt_ambient_pressure float,
         cloud_cover float,
         sea_level_pressure  float,
         gt_energy_yield float
) ROW FORMAT DELIMITED FIELDS TERMINATED BY ',' LINES TERMINATED BY '\n' LOCATION 's3://sagemaker-us-east-1-483525480215/CCPP/data/feature_set'
TBLPROPERTIES ('skip.header.line.count'='1')


In [11]:
pd.read_sql(statement, conn)

/tmp/ipykernel_18614/618921652.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql(statement, conn)


""


In [15]:
statement = """SELECT * FROM {}.{} LIMIT 960""".format(
    database_name, table_name_csv
)

print(statement)

SELECT * FROM ccpp_aws_fp.final_data_csv LIMIT 960


In [16]:
df = pd.read_sql(statement, conn)
df

/tmp/ipykernel_18614/8222894.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(statement, conn)


,gt_comp_dis_pressure,gt_exhaust_pressure,gt_inlet_temp,gt_air_filter_diff_pressure,solar_radiation,solar_energy,uv_index,gt_ambient_pressure,cloud_cover,sea_level_pressure,gt_energy_yield
0,10.564,19.166,1043.6,2.3927,0.0,0.0,0.0,1020.1,90.0,1023.0,110.16
1,10.572,19.119,1039.9,2.3732,0.0,0.0,0.0,1019.9,89.0,1023.0,109.23
2,10.543,19.178,1041.0,2.3854,0.0,0.0,0.0,1019.8,91.2,1023.1,109.62
3,10.458,19.180,1037.1,2.3825,0.0,0.0,0.0,1019.3,90.3,1022.3,108.88
4,10.464,19.206,1043.5,2.4044,0.0,0.0,0.0,1019.1,90.3,1022.2,110.09
...,...,...,...,...,...,...,...,...,...,...,...
955,12.053,25.659,1079.9,3.7007,465.4,1.7,5.0,1015.1,48.5,1020.3,133.51
956,12.014,25.959,1081.3,3.7301,471.0,1.7,5.0,1014.5,28.3,1020.1,134.50
957,12.078,26.047,1081.4,3.7174,403.3,1.5,4.0,1014.2,26.6,1019.3,134.55
958,12.071,25.716,1080.3,3.6579,277.5,1.0,3.0,1014.1,26.6,1019.3,133.66


In [17]:
statement = """SELECT count(1) FROM {}.{} LIMIT 100""".format(
    database_name, table_name_csv
)

print(statement)

SELECT count(1) FROM ccpp_aws_fp.final_data_csv LIMIT 100


In [18]:
df = pd.read_sql(statement, conn)
df

/tmp/ipykernel_18614/8222894.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(statement, conn)


,_col0
0,14764


In [24]:
%%html

<p><b>Shutting down your kernel for this notebook to release resources.</b></p>
<button class="sm-command-button" data-commandlinker-command="kernelmenu:shutdown" style="display:none;">Shutdown Kernel</button>
        
<script>
try {
    els = document.getElementsByClassName("sm-command-button");
    els[0].click();
}
catch(err) {
    // NoOp
}    
</script>